# PatelDelivers BV — ETL Pipeline
### Last-Mile Delivery Analytics | Netherlands | 2022–2024

**Author:** Harshilkumar Patel  
**University:** Wittenborg University of Applied Sciences, Netherlands  
**Stack:** Python · pandas · SQLAlchemy · MySQL 8.0

This notebook loads all 6 tables from the PatelDelivers BV Excel dataset into MySQL.  
Business analysis queries are in the `/sql` folder.

---

| Step | Action |
|---|---|
| 1 | Install and import libraries |
| 2 | Configure database connection |
| 3 | Create MySQL schema |
| 4 | Load all 6 tables with cleaning |
| 5 | Verify row counts |


---
## 1 — Install and Import Libraries


In [ ]:
# Install required libraries
import subprocess
subprocess.run(['pip', 'install', 'sqlalchemy', 'mysql-connector-python',
                'openpyxl', 'pandas', '--quiet'], capture_output=True)
print('Libraries ready')


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

print('pandas:     ', pd.__version__)
import sqlalchemy
print('sqlalchemy: ', sqlalchemy.__version__)


---
## 2 — Configuration

Update the two variables below before running:
- `EXCEL_PATH` — path to your local copy of the dataset
- MySQL credentials if different from defaults


In [ ]:
# ── DATABASE CREDENTIALS ──────────────────────────
# Replace with your own MySQL username and password
MYSQL_USER     = 'your_username'
MYSQL_PASSWORD = 'your_password'
MYSQL_HOST     = 'localhost'
MYSQL_SCHEMA   = 'swiftroute'

# ── DATASET PATH ──────────────────────────────────
# Update this to your local file path
# Windows example: r'C:\Users\YourName\Downloads\SwiftRoute_BV_Dataset.xlsx'
# Mac/Linux example: '/Users/yourname/Downloads/SwiftRoute_BV_Dataset.xlsx'
EXCEL_PATH = r'path/to/SwiftRoute_BV_Dataset.xlsx'


---
## 3 — Create MySQL Schema


In [ ]:
# Connect to MySQL and create the swiftroute schema
base_conn = f'mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}'
engine_root = create_engine(base_conn)

with engine_root.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {MYSQL_SCHEMA}'))
    conn.commit()

# Reconnect pointing to the swiftroute schema
engine = create_engine(f'{base_conn}/{MYSQL_SCHEMA}')
print(f'Connected to schema: {MYSQL_SCHEMA}')


---
## 4 — Load Tables


In [ ]:
# Open the Excel file — holds all 7 sheets
xl = pd.ExcelFile(EXCEL_PATH)
print('Sheets found:', xl.sheet_names)


### Table 1 — `dim_postcodes`
99 rows · PC4 postcode zones for all 5 cities · No column renames needed


In [ ]:
df_postcodes = pd.read_excel(xl, sheet_name='dim_postcodes')

print('Shape:  ', df_postcodes.shape)
print('Nulls:'); print(df_postcodes.isnull().sum())
display(df_postcodes.head(3))


In [ ]:
df_postcodes.to_sql(
    name='dim_postcodes',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'dim_postcodes loaded: {len(df_postcodes)} rows')


### Table 2 — `dim_drivers`
80 rows · Driver profiles · Column renames: `name` → `driver_name`, `active` → `is_active`


In [ ]:
df_drivers = pd.read_excel(xl, sheet_name='dim_drivers')

# Rename columns — 'name' and 'active' conflict with MySQL conventions
df_drivers = df_drivers.rename(columns={
    'name':   'driver_name',
    'active': 'is_active'
})

print('Shape:  ', df_drivers.shape)
print('Columns:', df_drivers.columns.tolist())
display(df_drivers.head(3))


In [ ]:
df_drivers.to_sql(
    name='dim_drivers',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'dim_drivers loaded: {len(df_drivers)} rows')


### Table 3 — `fact_deliveries` (Core Table)
24,923 rows · Every shipment attempt · Column renames: `month` → `month_name`, `year` → `year_num`, `year_month` → `yr_month`


In [ ]:
df_deliveries = pd.read_excel(xl, sheet_name='fact_deliveries')

# Rename MySQL reserved word columns
df_deliveries = df_deliveries.rename(columns={
    'month':      'month_name',   # MONTH is a MySQL function
    'year':       'year_num',     # YEAR is a MySQL function
    'year_month': 'yr_month'      # YEAR part triggers reserved word
})

# Quick data quality check
print('Shape:   ', df_deliveries.shape)
print('Dates:   ', df_deliveries['dispatch_date'].min(), 'to', df_deliveries['dispatch_date'].max())
print('Cities:  ', sorted(df_deliveries['city'].unique().tolist()))
print('Nulls in key columns:')
print(df_deliveries[['shipment_id','city','attempt_1_status',
                      'delivery_cost_eur','co2_emission_kg']].isnull().sum())


In [ ]:
print('Loading fact_deliveries — largest table, please wait...')

df_deliveries.to_sql(
    name='fact_deliveries',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'fact_deliveries loaded: {len(df_deliveries):,} rows')


### Table 4 — `warehouse_inventory`
40 rows · Stock snapshot across 5 warehouses · Column rename: `status` → `stock_status`


In [ ]:
df_warehouse = pd.read_excel(xl, sheet_name='warehouse_inventory')

# Rename — STATUS is a MySQL reserved word
df_warehouse = df_warehouse.rename(columns={'status': 'stock_status'})

print('Shape:  ', df_warehouse.shape)
print('Columns:', df_warehouse.columns.tolist())
display(df_warehouse.head(3))


In [ ]:
df_warehouse.to_sql(
    name='warehouse_inventory',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'warehouse_inventory loaded: {len(df_warehouse)} rows')


### Table 5 — `daily_volume_forecast`
4,960 rows · Daily volumes per city · Column renames: `date` → `forecast_date`, `month` → `month_name`, `year` → `year_num`
Note: blank cells in `peak_event` load as NULL — correct behaviour.


In [ ]:
df_forecast = pd.read_excel(xl, sheet_name='daily_volume_forecast')

# Three reserved word renames
df_forecast = df_forecast.rename(columns={
    'date':  'forecast_date',  # DATE is a MySQL data type keyword
    'month': 'month_name',     # MONTH is a MySQL function
    'year':  'year_num'        # YEAR is a MySQL function
})

print('Shape:  ', df_forecast.shape)
print('Columns:', df_forecast.columns.tolist())
print('Peak events:', df_forecast['peak_event'].value_counts(dropna=False).head(5).to_dict())


In [ ]:
df_forecast.to_sql(
    name='daily_volume_forecast',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'daily_volume_forecast loaded: {len(df_forecast):,} rows')


### Table 6 — `kpi_summary`
180 rows · Monthly KPIs per city (36 months × 5 cities) · No column renames needed


In [ ]:
df_kpi = pd.read_excel(xl, sheet_name='kpi_summary')

print('Shape:  ', df_kpi.shape)
print('Columns:', df_kpi.columns.tolist())
display(df_kpi.head(3))


In [ ]:
df_kpi.to_sql(
    name='kpi_summary',
    con=engine,
    if_exists='replace',
    index=False
)
print(f'kpi_summary loaded: {len(df_kpi)} rows')


---
## 5 — Verify All Tables


In [ ]:
# Read row counts back from MySQL to confirm all tables loaded correctly
verify_sql = '''
SELECT 'dim_postcodes'       AS table_name, COUNT(*) AS row_count FROM dim_postcodes
UNION ALL SELECT 'dim_drivers',                        COUNT(*) FROM dim_drivers
UNION ALL SELECT 'fact_deliveries',                    COUNT(*) FROM fact_deliveries
UNION ALL SELECT 'warehouse_inventory',               COUNT(*) FROM warehouse_inventory
UNION ALL SELECT 'daily_volume_forecast',             COUNT(*) FROM daily_volume_forecast
UNION ALL SELECT 'kpi_summary',                        COUNT(*) FROM kpi_summary
'''

df_verify = pd.read_sql(verify_sql, engine)

expected = {
    'dim_postcodes': 99, 'dim_drivers': 80,
    'fact_deliveries': 24923, 'warehouse_inventory': 40,
    'daily_volume_forecast': 4960, 'kpi_summary': 180
}

print('=' * 48)
print('  VERIFICATION — Row counts in MySQL')
print('=' * 48)
all_ok = True
for _, row in df_verify.iterrows():
    exp = expected.get(row['table_name'], 0)
    ok  = row['row_count'] == exp
    if not ok: all_ok = False
    status = 'OK     ' if ok else 'MISMATCH'
    print(f'  {status}  {row["table_name"]:<26} {row["row_count"]:>6} rows')

print('-' * 48)
print(f'  {"TOTAL":<34} {df_verify["row_count"].sum():>6} rows')
print('=' * 48)
print()
if all_ok:
    print('All tables match expected counts')
    print('Database ready -- run views_create.sql next in MySQL Workbench')
else:
    print('One or more row counts do not match -- check the loading steps above')


---
## Next Steps

Once all tables are verified:

1. Open MySQL Workbench
2. Run `sql/views_create.sql` — creates all 5 analytical views
3. Open Power BI Desktop → Get Data → MySQL Database → `localhost` / `swiftroute`
4. Load all 6 tables and 5 views

Business analysis queries are in the `/sql` folder:
- `Q1_city_fail_analysis.sql`
- `Q2_fail_reason_analysis.sql`
- `Q3_vehicle_co2_analysis.sql`
- `Q4_postcode_risk_analysis.sql`
- `Q5_business_impact.sql`

---
*PatelDelivers BV — Last-Mile Delivery Analytics*  
*Harshilkumar Patel · Wittenborg University of Applied Sciences · Netherlands 2025*
